Run this only if you're training on colab. 
Make sure you have added the shared drive as a shortcut to the **root** of your google drive.

If you're training your model offline and this cell throws `SyntaxError`, just ignore it.

In [51]:
import sys
import os

import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [3]:
currentlyRunningOnColab = 'google.colab' in sys.modules
if currentlyRunningOnColab:
  # Mount your own Google Drive
  from google.colab import drive
  drive.mount('/content/drive', force_remount=True)
  print("Mounted Google drive of current user")

  %cd /content
  print("Changed current directory to /content")

  # Mount github repository
  !git clone https://github.com/beazt123/AI-Project2021SUTD.git
  print("\nCloned repo onto current instance of machine")
  
  %cd AI-Project2021SUTD
  print("\nCurrently operating in the root directory of the Git repository")

  # dataFolderName = "aiproject"
  GITHUB_REPO_NAME = "AI-Project2021SUTD"
  codePath = GITHUB_REPO_NAME
  # /content/drive/.shortcut-targets-by-id/1OU-Ua5tGwc7PDL_nf0s1CPKoIQduRW5q/50.021 AI Project

Mounted at /content/drive
Mounted Google drive of current user
/content
Changed current directory to /content
Cloning into 'AI-Project2021SUTD'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 19 (delta 4), reused 0 (delta 0), pack-reused 0
Unpacking objects: 100% (19/19), done.

Cloned repo onto current instance of machine
/content/AI-Project2021SUTD

Currently operating in the root directory of the Git repository


In [4]:
# Change this variable if u wanna train offline
dataFolderPath = os.path.join("/content",
                              "drive", 
                              ".shortcut-targets-by-id", 
                              "19icV-F9BFrur0fxo4XvAZ82qIhipVrb-",
                              "aiProjectData50021")
# /content/drive/.shortcut-targets-by-id/19icV-F9BFrur0fxo4XvAZ82qIhipVrb-/aiProjectData50021

# Preparing the data

In [15]:
trainingPercentage = 0.7
testPercentage = 0.3

In [34]:
folders = ["COVID DATASET 1","COVID DATASET 2"]#, "COVID DATASET 3"]
csvLists = [glob.glob(f"{dataFolderPath}/{folder}/*.csv") for folder in folders]

In [36]:
overallTrain = []
overallTest = []
for csvList in csvLists:  
  train, test = train_test_split(csvList,
                                test_size=testPercentage,
                                train_size=trainingPercentage)
  overallTrain.extend(train)
  overallTest.extend(test)

In [110]:
def preProcessFunc(df):
  df_cp = df.copy()
  return df_cp[["followers",
                "friends",
                "retweets",
                "positive_sentiment",
                "negative_sentiment",
                "date.hour",
                "no_hashtag",
                "no_mentions"]]


class TwitterDataset(Dataset):
  def __init__(self, filenames):
    # `filenames` is a list of strings the contains all file names.
    # `batch_size` is the determines the number of files that we want to read in a chunk.
        self.filenames = filenames
  def __len__(self):
        return len(self.filenames)
  def __getitem__(self, idx): #idx means index of the chunk.
    # In this method, we do all the preprocessing.
    # First read data from files in a chunk. Preprocess it. Extract labels. Then return data and labels.
        csvFile = self.filenames[idx]
        df = pd.read_csv(csvFile)
        # if preProcessFunc:
        df = preProcessFunc(df)

        x_arr = torch.Tensor(df.drop(columns=['retweets']).to_numpy().astype(float))
        y = torch.Tensor(df["retweets"].to_numpy().astype(float))
        X = torch.squeeze( x_arr )
        if idx == self.__len__():  
          raise IndexError

        return X, y

In [111]:
train_loader = DataLoader(dataset = TwitterDataset(overallTrain),
                            batch_size = 1,
                            shuffle = True)

# Build the network
Make sure this network takes in whatever input you give it and outputs a number. Alternative, if this is an immediate model, like the zero/more than zero retweets classifier, than train it and save the parameters externally. Then train the regressor in another copy of this script.

In [ ]:
class NeuralNetwork:
  def __init__(self):
    pass

# Train the network
## Training parameters

In [ ]:
numEpochs = 1000
lr_rate = 0.01

model = NeuralNetwork()

loss_function = nn.MSELoss() # Change to BCELoss for classification problem
optimizer = torch.optim.SGD(model.parameters(), lr=lr_rate)

## Training for-loop

In [113]:
for i in numEpochs:
  for X, y in train_loader:
    X = torch.squeeze(X)
    y = y.T
    y_hat = model(X)

    optimizer.zero_grad() 
    y_hat = model(X)

    loss = loss_function(y_hat, y) #calculate the loss
    loss.backward() #backprop
    optimizer.step() #does the update

    if i % 500 == 0:
        print ("Epoch: {0}, Loss: {1}, ".format(i, loss.data.numpy()))

torch.Size([10000, 7])
10000
torch.Size([10000, 1])


## Evaluate the model

In [ ]:

test_loader = DataLoader(dataset = TwitterDataset(overallTest),
                            batch_size = 1,
                            shuffle = True)
cumLoss = 0
for (i, (X, y)) in enumerate(test_loader):
  X = torch.squeeze(X)
  y = y.T
  y_hat = model(X)
  # cum_loss += loss_fn(scores, labels).item()
  loss = loss_function(y_hat, y)
  cumLoss += loss

print(f"MSELoss: {cumLoss / len(test_loader)}")

# Save your model

In [ ]:
PATH = "model.pt" #change this name to the name of your network

torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
            }, PATH)

# Load model from elsewhere

In [ ]:
modelA = NeuralNetwork()
optimModelA = optim.SGD(modelA.parameters(), lr=0.001, momentum=0.9)

checkpoint = torch.load(PATH)

modelA.load_state_dict(checkpoint['model_state_dict'])
optimizerA.load_state_dict(checkpoint['optimizer_state_dict'])

modelA.eval()
# - or -
modelA.train()